# Workday Org Data — Lander (Fabric)

Lands a **Workday** worker extract dropped in the Lakehouse at `Files/org_workday/` and gets it ready
for the ValueLens PBIT's **`Chat + Agent Org Data`** query, which reads Delta table
`dbo.copilot_org_data`.

```
Workday export (CSV)  ->  Files/org_workday/  ->  this notebook  ->  dbo.copilot_org_data
```

## Why this notebook exists

The Workday extract is keyed on **`primaryWorkEmail`** — which is exactly the identity the model joins
on (`PersonId_Normalized` = `lower(trim(PersonId))`). But a Workday worker extract carries **no
manager, no display name and no AAD object id**, so it cannot on its own populate the org-hierarchy
visuals.

`Copilot_Org_Data_Direct_Ingester` (Graph `/users`) *does* produce those. So the default here is
**`MODE = 'enrich'`**: read the Entra-sourced snapshot, overlay the richer Workday HR attributes onto
it by email, and write the combined result back. You keep `managerUPN`, `displayName`, `OrgLevel`,
`HierarchyPath`, `IsManager` and `DirectReports` *and* gain Job Family, Persona, Worker Type and
Compensation Grade as slicers.

`MODE = 'standalone'` is available when Workday is your only org source. It writes a valid table, but
the manager-hierarchy columns come through blank.

## Run order

| Step | Notebook |
|---|---|
| 1 | `Copilot_Org_Data_Direct_Ingester` — writes the Entra baseline |
| 2 | **this notebook** — overlays Workday HR attributes |
| 3 | Refresh the semantic model |

> Run this **after** the Graph ingester, every time. The ingester overwrites `copilot_org_data`, so a
> refresh of the baseline drops the Workday columns until this notebook runs again.

## Expected Workday columns

`primaryWorkEmail` is the only required column — everything else is optional and loaded blank if the
export doesn't carry it. Header matching ignores case, spaces, underscores and punctuation, so
`Job Family Group`, `job_family_group` and `JobFamilyGroup` all resolve.

| Workday column | Becomes |
|---|---|
| `primaryWorkEmail` | `PersonId` / join key |
| `Job_Profile` | `JobTitle` |
| `Job_Family_Group` | `Organization` *(configurable)* + `Function` |
| `sub_Country` | `officeLocation` + `Location` |
| `country` | `country` |
| `On_Leave` | kept, plus derived `IsOnLeave` |
| `Job_Family`, `Persona`, `Compensation_Grade`, `Worker_Type`, `Worker_SubType` | kept as-is (new slicers) |

Any column the export carries that isn't listed above is **kept as-is** — nothing is dropped.

**How to use:** drop the export at `Files/org_workday/` (single CSV or several with identical
headers), attach this notebook to your Lakehouse, and Run all.


## 1. Configuration

In [ ]:
# === CONFIG ===
SOURCE_PATH  = 'Files/org_workday/'      # folder of Workday CSVs, or a single 'Files/org_workday/workers.csv'
OUTPUT_TABLE = 'dbo.copilot_org_data'    # table the PBIT's "Chat + Agent Org Data" query reads
BASE_TABLE   = 'dbo.copilot_org_data'    # Entra snapshot from Copilot_Org_Data_Direct_Ingester (MODE='enrich')

# 'enrich'     - overlay Workday attributes onto BASE_TABLE, keeping manager hierarchy + displayName.
# 'standalone' - Workday is the only org source; hierarchy columns are emitted blank.
MODE = 'enrich'

# Which Workday column drives the dashboard's main "Organization" (department) dimension.
# Job_Family_Group is the closest analogue to an Entra Department. Swap for 'Persona',
# 'Job_Family' or any other column in your export to reshape every org breakdown in the report.
ORGANIZATION_SOURCE = 'Job_Family_Group'

# Which source wins where BOTH Entra and Workday supply a value (Organization, JobTitle,
# country, officeLocation). 'workday' treats the HR system as the record of truth.
ATTRIBUTE_PRECEDENCE = 'workday'         # 'workday' | 'entra'

# Workers present in Workday but absent from Entra. False keeps the Entra population intact
# and just reports the gap; True adds them as new rows (no hierarchy, no displayName).
INCLUDE_UNMATCHED_WORKDAY = False

# Guard against landing the wrong file / wrong tenant's export: MODE='enrich' fails if fewer
# than this share of Workday rows match an Entra identity.
MIN_WORKDAY_MATCH_RATE = 0.5

ALLOW_EMPTY_SNAPSHOT = False             # True only for an intentional empty first install

## 2. Locate and read the Workday export

A missing export is **not** an error — it leaves any existing `copilot_org_data` snapshot untouched so
a failed hand-off never blanks the dashboard.

In [ ]:
import re
import notebookutils
from pyspark.sql import functions as F

if MODE not in ('enrich', 'standalone'):
    raise ValueError(f"MODE must be 'enrich' or 'standalone', got {MODE!r}.")
if ATTRIBUTE_PRECEDENCE not in ('workday', 'entra'):
    raise ValueError(f"ATTRIBUTE_PRECEDENCE must be 'workday' or 'entra', got {ATTRIBUTE_PRECEDENCE!r}.")
if not 0 <= MIN_WORKDAY_MATCH_RATE <= 1:
    raise ValueError('MIN_WORKDAY_MATCH_RATE must be between 0 and 1.')


def _safe_ls(path):
    try:
        return list(notebookutils.fs.ls(path))
    except Exception as exc:
        lowered = str(exc).lower()
        if 'not found' in lowered or 'no such file' in lowered or 'does not exist' in lowered:
            return []
        raise


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


def _resolve_source(path):
    """Return the list of CSV paths to read, or [] when nothing is staged."""
    if path.endswith('/'):
        files = [item for item in _safe_ls(path)
                 if not item.isDir and item.name.lower().endswith(('.csv', '.txt'))]
        return sorted(item.path for item in files)
    folder, name = path.rsplit('/', 1)
    return [path] if any(item.name == name for item in _safe_ls(folder)) else []


sources = _resolve_source(SOURCE_PATH)

if not sources:
    print(f'No Workday export found at {SOURCE_PATH} - nothing landed.')
    if _table_exists(OUTPUT_TABLE):
        print(f'Preserving the existing {OUTPUT_TABLE} snapshot unchanged.')
    elif not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError(
            f'No Workday export at {SOURCE_PATH} and {OUTPUT_TABLE} does not exist yet. '
            'Drop the export, or set ALLOW_EMPTY_SNAPSHOT = True for an intentional empty first install.'
        )
else:
    print(f'Reading {len(sources)} file(s):')
    for item in sources:
        print(f'  {item}')

    wd = (spark.read
          .option('header', True)
          .option('multiLine', True)
          .option('escape', '"')
          .option('encoding', 'UTF-8')
          .csv(sources))

    # Strip the UTF-8 BOM Workday writes ahead of the first header, plus stray padding.
    for column in wd.columns:
        cleaned = column.replace('\ufeff', '').strip()
        if cleaned != column:
            wd = wd.withColumnRenamed(column, cleaned)

    print(f'\nRaw rows: {wd.count():,} | columns: {wd.columns}')

## 3. Resolve Workday headers to the canonical schema

Canonical names are added as **copies**, never renames, so no source column is consumed — every
Workday field stays available to the report even after it has been mapped.

In [ ]:
if sources:
    def _norm(value):
        return re.sub(r'[^a-z0-9]', '', str(value).lower())

    # target -> accepted source headers, most specific first
    ALIAS_PLAN = {
        'primaryWorkEmail':   ['primaryWorkEmail', 'Primary Work Email', 'Work Email', 'Email',
                               'Email Address', 'Work Email Address', 'userPrincipalName', 'UPN'],
        'Job_Profile':        ['Job_Profile', 'Job Profile', 'Job Title', 'JobTitle', 'Business Title', 'Position'],
        'Job_Family':         ['Job_Family', 'Job Family'],
        'Job_Family_Group':   ['Job_Family_Group', 'Job Family Group'],
        'Persona':            ['Persona', 'Worker Persona'],
        'Compensation_Grade': ['Compensation_Grade', 'Compensation Grade', 'Grade', 'Pay Grade'],
        'Worker_Type':        ['Worker_Type', 'Worker Type', 'Employee Type'],
        'Worker_SubType':     ['Worker_SubType', 'Worker Sub Type', 'Worker Subtype', 'Employee Sub Type'],
        'On_Leave':           ['On_Leave', 'On Leave', 'Leave of Absence', 'OnLeave'],
        'country':            ['country', 'Country', 'Location Country', 'Country/Region'],
        'sub_Country':        ['sub_Country', 'Sub Country', 'Sub-Country', 'Region', 'State', 'Province'],
    }

    # Reject headers that collide once normalized - Spark resolves names case-insensitively,
    # so 'Country' + 'country' would otherwise become an ambiguous reference at select time.
    groups = {}
    for column in wd.columns:
        groups.setdefault(_norm(column), []).append(column)
    clashes = [cols for cols in groups.values() if len(cols) > 1]
    if clashes:
        raise ValueError(f'Ambiguous Workday headers after normalization: {clashes}')

    lookup = {_norm(column): column for column in wd.columns}
    resolved, fallback, missing = {}, [], []
    for target, candidates in ALIAS_PLAN.items():
        for candidate in candidates:
            hit = lookup.get(_norm(candidate))
            if hit is not None:
                resolved[target] = hit
                if hit != target:
                    fallback.append((target, hit))
                break
        else:
            missing.append(target)

    if 'primaryWorkEmail' not in resolved:
        raise ValueError(
            'Workday export has no work-email column. It is the only required field - it is the '
            f'identity the model joins on. Headers seen: {wd.columns}'
        )
    if ORGANIZATION_SOURCE not in resolved and ORGANIZATION_SOURCE not in wd.columns:
        raise ValueError(
            f'ORGANIZATION_SOURCE={ORGANIZATION_SOURCE!r} is not present in the export. '
            f'Pick one of: {wd.columns}'
        )

    print('Workday schema match')
    print(f'  exact     : {len(resolved) - len(fallback)}')
    if fallback:
        print(f'  via alias : {len(fallback)}')
        for target, source in fallback:
            print(f'              {target!r} <- {source!r}')
    if missing:
        print(f'  NOT FOUND : {len(missing)} (loaded blank)')
        for target in missing:
            print(f'              {target}')
    extras = [c for c in wd.columns if c not in set(resolved.values())]
    if extras:
        print(f'  extra cols: {len(extras)} kept as-is -> {extras}')

## 4. Project to canonical columns, validate identities, de-duplicate

A blank email is an unusable identity and a conflicting duplicate is a silent data-quality failure —
both stop the run rather than shipping a subtly wrong org dimension.

In [ ]:
if sources:
    def _blank_to_null(column):
        trimmed = F.trim(F.col(column).cast('string'))
        return F.when(trimmed == '', None).otherwise(trimmed)

    projected = wd
    for target, source in resolved.items():
        if source != target:
            projected = projected.withColumn(target, F.col(f'`{source}`'))
    for target in missing:
        projected = projected.withColumn(target, F.lit(None).cast('string'))
    for target in ALIAS_PLAN:
        projected = projected.withColumn(target, _blank_to_null(target))

    org_col = ORGANIZATION_SOURCE if ORGANIZATION_SOURCE in projected.columns else resolved[ORGANIZATION_SOURCE]
    projected = (projected
                 .withColumn('_wd_join', F.lower(F.trim(F.col('primaryWorkEmail'))))
                 .withColumn('Organization', _blank_to_null(org_col))
                 .withColumn('JobTitle', F.col('Job_Profile'))
                 .withColumn('Function', F.col('Job_Family_Group'))
                 .withColumn('officeLocation', F.col('sub_Country'))
                 .withColumn('Location', F.col('sub_Country'))
                 .withColumn('IsOnLeave',
                             F.when(F.col('On_Leave').isNull(), None)
                              .when(F.lower(F.col('On_Leave')).isin('1', 'true', 'yes', 'y'), F.lit('TRUE'))
                              .otherwise(F.lit('FALSE'))))

    blank_ids = projected.filter(F.col('_wd_join').isNull() | (F.col('_wd_join') == '')).count()
    if blank_ids:
        raise ValueError(
            f'{blank_ids:,} Workday row(s) have a blank work email. Refusing to write anonymous '
            'identities - fix the export or filter those workers out at source.'
        )

    # Exact-duplicate rows collapse silently; genuinely conflicting records must be resolved upstream.
    compare_cols = [c for c in projected.columns if c != '_wd_join']
    deduped = projected.dropDuplicates(['_wd_join'] + compare_cols)
    conflicts = (deduped.groupBy('_wd_join').count().filter(F.col('count') > 1))
    conflict_count = conflicts.count()
    if conflict_count:
        print('Conflicting Workday rows (same email, different attributes):')
        conflicts.show(10, truncate=False)
        raise ValueError(
            f'{conflict_count:,} work email(s) appear more than once with different attribute values. '
            'Refusing to pick a winner arbitrarily - de-duplicate the Workday export first.'
        )

    exact_dupes = projected.count() - deduped.count()
    if exact_dupes:
        print(f'Collapsed {exact_dupes:,} exact-duplicate row(s).')

    projected = deduped
    workday_rows = projected.count()
    if workday_rows == 0:
        raise ValueError('Workday export parsed 0 rows; refusing to write an empty snapshot.')
    print(f'Validated Workday rows: {workday_rows:,}')

## 5. Combine with the Entra baseline (`MODE = 'enrich'`)

Left-joins the Workday attributes onto the Graph snapshot by normalised email. Overlay columns
(`Organization`, `JobTitle`, `country`, `officeLocation`) are **coalesced**, not blindly replaced, so a
worker missing from the Workday file keeps their Entra value instead of going blank.

Re-running is safe: Workday-only columns are dropped from the baseline before the join, so the
notebook is idempotent whether or not the Graph ingester has refreshed in between.

In [ ]:
if sources:
    HIER_FIXED  = ['OrgLevel', 'HierarchyPath', 'TopOfChain_Name', 'IsManager', 'DirectReports']
    HIER_LEVELS = [f'Level{i}_Name' for i in range(15)]
    OVERLAY     = ['Organization', 'JobTitle', 'country', 'officeLocation']
    PASSTHROUGH = ['Job_Profile', 'Job_Family', 'Job_Family_Group', 'Persona', 'Compensation_Grade',
                   'Worker_Type', 'Worker_SubType', 'On_Leave', 'IsOnLeave', 'sub_Country',
                   'Function', 'Location', 'primaryWorkEmail'] + extras
    # An export carrying its own 'Function'/'Location' header would land in extras too.
    PASSTHROUGH = list(dict.fromkeys(PASSTHROUGH))

    if MODE == 'enrich':
        if not _table_exists(BASE_TABLE):
            raise ValueError(
                f"MODE='enrich' needs {BASE_TABLE}, which does not exist. Run "
                "Copilot_Org_Data_Direct_Ingester first, or set MODE='standalone'."
            )

        base = spark.table(BASE_TABLE)
        if 'PersonId' not in base.columns:
            raise ValueError(f'{BASE_TABLE} has no PersonId column; it is not a valid org snapshot.')

        # Idempotency: shed any columns a previous enrich run added before re-joining.
        base = base.drop(*[c for c in PASSTHROUGH if c in base.columns])
        base = base.withColumn('_base_join', F.lower(F.trim(F.col('PersonId').cast('string'))))

        base_rows = base.count()
        matched = base.join(projected.select('_wd_join'), base['_base_join'] == F.col('_wd_join'), 'inner').count()
        match_rate = matched / workday_rows if workday_rows else 0.0
        print(f'Entra rows: {base_rows:,} | Workday rows: {workday_rows:,} | matched: {matched:,} '
              f'({match_rate:.1%} of Workday)')
        if match_rate < MIN_WORKDAY_MATCH_RATE:
            raise ValueError(
                f'Only {match_rate:.1%} of Workday rows matched an Entra identity, below '
                f'MIN_WORKDAY_MATCH_RATE={MIN_WORKDAY_MATCH_RATE:.0%}. This usually means the wrong '
                'export, the wrong tenant, or an email-domain mismatch between Workday and Entra.'
            )

        wd_side = projected.select(
            F.col('_wd_join'),
            *[F.col(c).alias(f'_wd_{c}') for c in OVERLAY if c in projected.columns],
            *[F.col(c) for c in PASSTHROUGH if c in projected.columns],
        )

        how = 'full_outer' if INCLUDE_UNMATCHED_WORKDAY else 'left'
        joined = base.join(wd_side, base['_base_join'] == wd_side['_wd_join'], how)

        for column in OVERLAY:
            wd_ref = F.col(f'_wd_{column}') if f'_wd_{column}' in joined.columns else F.lit(None).cast('string')
            base_ref = F.col(column) if column in base.columns else F.lit(None).cast('string')
            first, second = (wd_ref, base_ref) if ATTRIBUTE_PRECEDENCE == 'workday' else (base_ref, wd_ref)
            joined = joined.withColumn(column, F.coalesce(first, second))
        joined = joined.drop(*[f'_wd_{c}' for c in OVERLAY if f'_wd_{c}' in joined.columns])

        if INCLUDE_UNMATCHED_WORKDAY:
            # Workday-only workers have no Entra identity - seed PersonId/id from the work email
            # so they still carry a usable join key into the interactions fact.
            joined = (joined
                      .withColumn('PersonId', F.coalesce(F.col('PersonId'), F.col('primaryWorkEmail')))
                      .withColumn('_base_join', F.coalesce(F.col('_base_join'), F.col('_wd_join'))))
            if 'id' in joined.columns:
                joined = joined.withColumn('id', F.coalesce(F.col('id'), F.col('PersonId')))
        else:
            unmatched = workday_rows - matched
            if unmatched:
                print(f'{unmatched:,} Workday worker(s) have no Entra identity and were not added. '
                      'Set INCLUDE_UNMATCHED_WORKDAY = True to include them.')

        result = joined.drop('_wd_join', '_base_join')

    else:
        # Standalone: Workday is the only source. Emit the Entra-shaped columns so the table stays
        # schema-compatible with the Graph ingester's output; the ones Workday cannot supply are
        # typed nulls rather than missing columns, so the PBIT binds cleanly and renders blank.
        result = projected.withColumn('PersonId', F.col('primaryWorkEmail'))
        result = result.withColumn('id', F.col('PersonId'))
        result = result.withColumn('displayName', F.lit(None).cast('string'))
        result = result.withColumn('companyName', F.lit(None).cast('string'))
        result = result.withColumn('city', F.col('sub_Country'))
        result = result.withColumn('managerUPN', F.lit(None).cast('string'))
        result = result.withColumn(
            'accountEnabled',
            F.when(F.col('IsOnLeave') == 'TRUE', F.lit('False')).otherwise(F.lit('True')))
        for column in HIER_FIXED + HIER_LEVELS:
            result = result.withColumn(column, F.lit(None).cast('string'))
        result = result.drop('_wd_join')
        print('MODE=standalone: manager hierarchy, displayName and companyName are emitted blank. '
              'Run Copilot_Org_Data_Direct_Ingester and switch to MODE=\'enrich\' to populate them.')

## 6. Finalise — normalised key, `TotalEmployees`, safe column names

`PersonId_Normalized` is the model's join key and `TotalEmployees` backs the adoption-rate
denominators. The PBIT adds both if absent, but computing them here keeps the Delta table
self-describing and lets the SQL endpoint answer the same questions.

In [ ]:
if sources:
    result = result.withColumn(
        'PersonId_Normalized',
        F.when(F.col('PersonId').isNull(), None)
         .otherwise(F.lower(F.trim(F.col('PersonId').cast('string')))))

    duplicate_people = (result
                        .filter(F.col('PersonId_Normalized').isNotNull() & (F.col('PersonId_Normalized') != ''))
                        .groupBy('PersonId_Normalized').count()
                        .filter(F.col('count') > 1)
                        .count())
    if duplicate_people:
        raise ValueError(
            f'{duplicate_people:,} duplicate PersonId_Normalized value(s) after the join. '
            'Refusing to write a fan-out that would double-count people in every org measure.'
        )

    total = result.count()
    if total == 0 and _table_exists(OUTPUT_TABLE):
        raise ValueError(f'Produced 0 rows; refusing to replace the existing {OUTPUT_TABLE}.')
    if total == 0 and not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError('Produced 0 rows and no existing snapshot; refusing to write an empty table.')

    result = result.withColumn('TotalEmployees', F.lit(str(total)))
    result = result.withColumn('OrgData_Source', F.lit(f'workday:{MODE}'))

    _INVALID = re.compile(r'[ ,;{}()\n\t=]')
    result = result.toDF(*[_INVALID.sub('_', c) for c in result.columns])

    print(f'Final rows: {total:,} | columns: {len(result.columns)}')
    print(result.columns)

## 7. Write to the Lakehouse

In `enrich` mode `OUTPUT_TABLE` and `BASE_TABLE` are normally the same table, and Spark cannot safely
overwrite a Delta table that the plan still reads from. Materialising through a staging table breaks
that lineage, so the read completes before the target is replaced.

In [ ]:
if sources:
    if OUTPUT_TABLE == BASE_TABLE and MODE == 'enrich':
        staging = f'{OUTPUT_TABLE}_workday_stage'
        (result.write.format('delta').mode('overwrite')
               .option('overwriteSchema', 'true').saveAsTable(staging))
        (spark.table(staging).write.format('delta').mode('overwrite')
              .option('overwriteSchema', 'true').saveAsTable(OUTPUT_TABLE))
        spark.sql(f'DROP TABLE IF EXISTS {staging}')
    else:
        (result.write.format('delta').mode('overwrite')
               .option('overwriteSchema', 'true').saveAsTable(OUTPUT_TABLE))

    print(f'Rows written to {OUTPUT_TABLE}: {spark.table(OUTPUT_TABLE).count():,}')

## 8. Verify

In [ ]:
if sources:
    tbl = spark.table(OUTPUT_TABLE)

    print('Sample:')
    tbl.select(*[c for c in ['PersonId', 'displayName', 'Organization', 'JobTitle',
                             'Persona', 'Worker_Type', 'managerUPN']
                 if c in tbl.columns]).show(10, truncate=False)

    print('Organization distribution:')
    tbl.groupBy('Organization').count().orderBy(F.desc('count')).show(20, truncate=False)

    # Coverage tells you how much of the org dimension the Workday file actually reached.
    for column in ['Persona', 'Job_Family_Group', 'Worker_Type', 'Compensation_Grade', 'managerUPN']:
        if column in tbl.columns:
            filled = tbl.filter(F.col(column).isNotNull() & (F.trim(F.col(column)) != '')).count()
            print(f'  {column:<20} populated on {filled:,} / {tbl.count():,} rows')

---
**Connect the PBIT**: `dbo.copilot_org_data` is consumed by the **`Chat + Agent Org Data`** query via
`FabricTable("copilot_org_data")`. Leave the `Org Data File` parameter blank when opening the
template — the query reads the Delta table through the Fabric SQL endpoint.

The query keeps every source column, so `Persona`, `Job_Family`, `Job_Family_Group`,
`Compensation_Grade`, `Worker_Type`, `Worker_SubType` and `IsOnLeave` all arrive in the model and can
be dropped straight onto a slicer. `Function` and `Location` are model-declared columns, so they
populate existing visuals with no report edit at all.
